<a href="https://colab.research.google.com/github/HudcovicMarcel/grassland-management-from-aerial-imagery/blob/main/GMAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grassland Management from Aerial Imagery (GMAI)

This notebook serves as supplementary material for the [GMAI dataset and repository](https://github.com/HudcovicMarcel/grassland-management-from-aerial-imagery).
It provides a **step-by-step demonstration** of the methodology used in the referenced study.

The notebook can be run using the **Demo Data and Model** available from the Zenodo repository (https://doi.org/10.5281/zenodo.24108529), which can be downloaded in the first step.

The demo data includes seven images and corresponding label files for each class in the dataset.
**Note:** We do **not recommend training** the model with this small demo dataset.
For actual training or reproducing results, please use the full `dataset.zip` from the repository.

**0.1. Demo data download**

Downloads demo data from Zenodo repository.

In [ ]:
import requests
import zipfile
import os
import shutil

demo_url = "https://zenodo.org/records/24108529/files/demo_data.zip?download=1"
model_url = "https://zenodo.org/records/24108529/files/YOLO11x_GMAI.pt?download=1"

output_dir = "GMAI"
demo_zip = "demo_data.zip"
model_file = os.path.join(output_dir, "YOLO11x_GMAI.pt")

def download(url, path):
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

download(demo_url, demo_zip)

os.makedirs(output_dir, exist_ok=True)
with zipfile.ZipFile(demo_zip, "r") as zip_ref:
    zip_ref.extractall(output_dir)

macosx_path = os.path.join(output_dir, "__MACOSX")
if os.path.exists(macosx_path):
    shutil.rmtree(macosx_path)

download(model_url, model_file)

print('Data downloaded')

**0.2. Install necessary packages**

In [ ]:
!pip install rasterio
!pip install ultralytics

# **1. Dataset creation**





**1.1. Importing prerequisites**

In [ ]:
import os
import rasterio
from rasterio.windows import Window
from collections import Counter
from PIL import Image
import cv2
import shutil
from collections import defaultdict
import random
import yaml


**1.2. Image tiling**

In [ ]:
img_path = '/content/GMAI/SAMPLE_ORTO' # Folder path with images
out_path = '/content/GMAI/SAMPLE_TILES' # Output folder for tiles

os.makedirs(out_path,exist_ok=True)

image_files = []

for dirpath, _, filenames in os.walk(img_path):
    for file in filenames:
        if file.lower().endswith(('.tif', '.tiff')): # Change according to your image file extension
            full_path = os.path.join(dirpath, file)
            image_files.append(full_path)
for image in image_files:
    with rasterio.open(image,"r+") as src:
        image_width = src.width
        image_height = src.height
        tiling_size = 1280  # Final size of image
        overlap = 0.1   # Overlap between images
        stride_x, stride_y = int(tiling_size * (1 - overlap)), int(tiling_size * (1 - overlap))

        tile_id=0

        for top in range(0, image_height, stride_y):
            for left in range(0, image_width, stride_x):
                if left + tiling_size > image_width:
                    left = image_width - tiling_size
                if top + tiling_size > image_height:
                    top = image_height - tiling_size
                window = Window(left, top, tiling_size, tiling_size)
                bounds = rasterio.windows.bounds(window, transform=src.transform)
                minX, minY, maxX, maxY = bounds
                tile_path = os.path.join(out_path, f"{os.path.basename(os.path.splitext(image)[0])}_{tile_id}.tif") #Tile name will contain name of image plus tile ID
                tile_id+=1
                tile_data = src.read(indexes=[1, 2, 3],window=window)
                with rasterio.open(
                                tile_path,
                                "w",
                                driver="GTiff",
                                height=tile_data.shape[1],
                                width=tile_data.shape[2],
                                count=3,
                                dtype=tile_data.dtype,
                                crs=src.crs,
                                transform=src.window_transform(window),
                            ) as dst:
                                dst.write(tile_data)

print("Created",(len(os.listdir(out_path))),'tiles from',(len(image_files)),'images')

**1.3. Data labeling**

All object annotations were created using the Label Studio graphical interface.  
We recommend running Label Studio on your local computer to label or review data, as it provides a user-friendly GUI and full control over the labeling process.
Further information on: https://labelstud.io/

**1.4. Split dataset by source tile and create YAML file for training**

Model tiles overlap only within their source tile (orthophotomosaic tile of 2.5 × 2 km), so whole source tiles are randomly assigned to the training, validation and test subsets (about 60/20/20). Overlapping tiles and, after the next step, augmented copies of the same tile therefore never occur in different subsets. In the study, the assignment of source tiles was additionally balanced by class (see the paper).

In [ ]:
images_folder = "/content/GMAI/SAMPLE_DATASET/images"
labels_folder = "/content/GMAI/SAMPLE_DATASET/labels"
output_folder = "/content/GMAI/SAMPLE_DATASET/final_dataset"

split_ratios = {"train": 0.6, "val": 0.2, "test": 0.2}
class_names = ["Cattle", "Haybale_round", "Haybale_square", "Haystack", "Horse", "Machinery", "Sheep"]
random.seed(0)

# Model tiles are named <source tile>_<tile id>
tiles = sorted(os.path.splitext(f)[0] for f in os.listdir(labels_folder) if f.endswith(".txt"))
source_of = {t: t.rsplit("_", 1)[0] for t in tiles}

# Random assignment of whole source tiles to the subsets
sources = sorted(set(source_of.values()))
random.shuffle(sources)
n_train = round(len(sources) * split_ratios["train"])
n_val = round(len(sources) * split_ratios["val"])
split_of_source = {s: "train" if i < n_train else "val" if i < n_train + n_val else "test"
                   for i, s in enumerate(sources)}

for split in split_ratios:
    for sub in ("images", "labels"):
        os.makedirs(os.path.join(output_folder, split, sub), exist_ok=True)

split_class_counts = {split: Counter() for split in split_ratios}
for t in tiles:
    split = split_of_source[source_of[t]]
    shutil.copy(os.path.join(images_folder, t + ".tif"), os.path.join(output_folder, split, "images", t + ".tif"))
    shutil.copy(os.path.join(labels_folder, t + ".txt"), os.path.join(output_folder, split, "labels", t + ".txt"))
    with open(os.path.join(labels_folder, t + ".txt")) as f:
        split_class_counts[split].update(int(line.split()[0]) for line in f if line.strip())

# Create YAML file for training
yaml_data = {
    "train": os.path.abspath(os.path.join(output_folder, "train")),
    "val": os.path.abspath(os.path.join(output_folder, "val")),
    "test": os.path.abspath(os.path.join(output_folder, "test")),
    "nc": len(class_names),
    "names": class_names
}
with open(os.path.join(output_folder, "dataset.yaml"), "w") as yaml_file:
    yaml.dump(yaml_data, yaml_file, default_flow_style=False)

for split, counts in split_class_counts.items():
    n_tiles = len(os.listdir(os.path.join(output_folder, split, "labels")))
    print(f"{split}: {n_tiles} tiles,", {class_names[c]: n for c, n in sorted(counts.items())})

**1.5. Data augmentation of the training subset**

Only the training subset is augmented; the validation and test subsets keep their original tiles. Classes are grouped by their number of bounding boxes in the training subset relative to the most frequent class:

* more than 50%: one copy, rotated by 180° (combined horizontal and vertical flip),
* 25–50%: three copies (180° rotation, horizontal flip, vertical flip),
* less than 25%: all seven copies (90° rotations and their flipped variants).

A tile containing several classes receives the copies of all of them. In the GMAI dataset, no class falls into the 25–50% group, so each training tile is present as 2 or 8 images. The suffixes of the copies are the same as in `dataset.zip`.

In [ ]:
train_images = os.path.join(output_folder, "train", "images")
train_labels = os.path.join(output_folder, "train", "labels")

# Pixel transform and the matching transform of YOLO boxes (normalised x, y, w, h) for each copy
TRANSFORMS = {
    "flipped":                (lambda im: cv2.rotate(im, cv2.ROTATE_180),
                               lambda x, y, w, h: (1 - x, 1 - y, w, h)),
    "flipped_hor":            (lambda im: cv2.flip(im, 1),
                               lambda x, y, w, h: (1 - x, y, w, h)),
    "flipped_ver":            (lambda im: cv2.flip(im, 0),
                               lambda x, y, w, h: (x, 1 - y, w, h)),
    "rotate_90":              (lambda im: cv2.rotate(im, cv2.ROTATE_90_CLOCKWISE),
                               lambda x, y, w, h: (1 - y, x, h, w)),
    "rotate_90_ccw":          (lambda im: cv2.rotate(im, cv2.ROTATE_90_COUNTERCLOCKWISE),
                               lambda x, y, w, h: (y, 1 - x, h, w)),
    "rotate_90_ccw_flip_hor": (lambda im: cv2.flip(cv2.rotate(im, cv2.ROTATE_90_COUNTERCLOCKWISE), 1),
                               lambda x, y, w, h: (1 - y, 1 - x, h, w)),
    "rotate_90_ccw_flip_ver": (lambda im: cv2.flip(cv2.rotate(im, cv2.ROTATE_90_COUNTERCLOCKWISE), 0),
                               lambda x, y, w, h: (y, x, h, w)),
}
COPIES = {"> 50%": ["flipped"],
          "25-50%": ["flipped", "flipped_hor", "flipped_ver"],
          "< 25%": list(TRANSFORMS)}

def read_yolo_labels(label_file):
    with open(label_file) as f:
        return [[int(p[0])] + [float(v) for v in p[1:5]] for p in (line.split() for line in f) if p]

# Class groups from the number of boxes in the training subset
originals = sorted(os.path.splitext(f)[0] for f in os.listdir(train_labels) if f.endswith(".txt"))
labels_of = {t: read_yolo_labels(os.path.join(train_labels, t + ".txt")) for t in originals}
class_counts = Counter(label[0] for labels in labels_of.values() for label in labels)
largest = max(class_counts.values())
group = {c: "> 50%" if n > 0.5 * largest else "25-50%" if n > 0.25 * largest else "< 25%"
         for c, n in class_counts.items()}
for c in sorted(class_counts):
    print(f"{class_names[c]}: {class_counts[c]} boxes ({class_counts[c] / largest:.0%}) -> {group[c]}")

# Save the copies next to the originals in the training subset
added = 0
for t in originals:
    suffixes = set()
    for c in {label[0] for label in labels_of[t]}:
        suffixes.update(COPIES[group[c]])
    if not suffixes:
        continue
    image = cv2.imread(os.path.join(train_images, t + ".tif"), cv2.IMREAD_UNCHANGED)
    for suffix, (transform_image, transform_box) in TRANSFORMS.items():
        if suffix not in suffixes:
            continue
        cv2.imwrite(os.path.join(train_images, f"{t}_{suffix}.tif"), transform_image(image))
        with open(os.path.join(train_labels, f"{t}_{suffix}.txt"), "w") as f:
            for c, *box in labels_of[t]:
                f.write(" ".join([str(c)] + [f"{v:.6f}" for v in transform_box(*box)]) + "\n")
        added += 1

print(f"Added {added} augmented copies to {len(originals)} training tiles; validation and test are not augmented.")

# **2. Image quality assessment (IQA)**

**2.1. Importing prerequisites**

In [ ]:
import glob
import pandas as pd
import random
import os
import tensorflow as tf
from PIL import Image
import numpy as np
from skimage.exposure import match_histograms
import skimage
import matplotlib.pyplot as plt
import seaborn as sns

**2.2. Random selection of samples**


In [ ]:
images_path = '/content/GMAI/SAMPLE_DATASET/images' # Path to images
labels_path = '/content/GMAI/SAMPLE_DATASET/labels/' # Path to label files

txts = glob.glob(f"{labels_path}/**.txt", recursive=True)

all_classes = [[] for _ in range(7)]
samples = []

for file in txts:
    data = pd.read_csv(file, sep=" ", header=None)
    class_ids = data[0].unique()

    for class_num in class_ids:
        if 0 <= class_num < 7:
            all_classes[int(class_num)].append(file)

# Pick random samples for each class
for x in all_classes:
    sample = random.sample(x,1) # Adjust number of samples for each class
    samples.append(sample)

img_x = []
img_y = []
img_name = []

# Filter samples at the edges of images
for class_num in range(7):
    count = 0
    for file in samples[class_num]:
        data = pd.read_csv(file, sep=" ", header=None)
        data = data[data[0] == class_num]
        data = data.sample(n=1)
        y = int(data[1].iloc[0] * 1280 - 30)
        x = int(data[2].iloc[0] * 1280 - 30)
        if y > 60 and y < 1220:
            if x > 60 and x < 1220:
                img_y.append(y)
                img_x.append(x)
                img_name.append(os.path.join(images_path, os.path.basename(os.path.splitext(file)[0]+'.tif')))

**2.3. Resizing and IQA metrics calculation**


In [ ]:
def IQA(image,x,y):
    image = Image.open(image)
    image = image.convert("RGB")
    image = np.array(image)

    crop_size = 60

    cropped = tf.image.crop_to_bounding_box(image, x, y, crop_size, crop_size) # Crops image around sample
    resized_20 = tf.image.resize(cropped, [int(crop_size * 0.75), int(crop_size * 0.75)], method = 'area')/255 # Resize image to correspond GSD of 20cm
    resized_25 = tf.image.resize(cropped, [int(crop_size * 0.6), int(crop_size * 0.6)], method = 'area')/255 # Resize image to correspond GSD of 25cm

    methods = ['nearest','bicubic','lanczos3','lanczos5','gaussian'] # Tested interpolation methods

    metrics = []

    for method in methods:
        globals()[f'resized_20_{method}'] = tf.image.resize(resized_20, [int(crop_size), int(crop_size)], method = method)
        globals()[f'resized_25_{method}'] = tf.image.resize(resized_25, [int(crop_size), int(crop_size)], method = method)

        img=globals()[f'resized_20_{method}'].numpy()
        ref=tf.image.convert_image_dtype(cropped, dtype=tf.float32).numpy()

        # Resize images from 20cm back to 15cm and compute IQA metrics
        matched_20 = match_histograms(img, ref, channel_axis= -1)

        SSIM_20 = tf.image.ssim(cropped/255, globals()[f'resized_20_{method}'], max_val=1.0,filter_size=9,
                              filter_sigma=1, k1=0.01, k2=0.03)
        SSIM_20_matched = tf.image.ssim(cropped/255, matched_20, max_val=1.0,filter_size=9,
                              filter_sigma=1, k1=0.01, k2=0.03)
        PSNR_20 = tf.image.psnr(cropped/255, globals()[f'resized_20_{method}'], max_val = 1.0)
        PSNR_20_matched = tf.image.psnr(cropped/255, matched_20, max_val = 1.0)
        MSE_20 = skimage.metrics.mean_squared_error(ref,img)
        MSE_20_matched = skimage.metrics.mean_squared_error(ref,matched_20)
        MAE_20 = tf.reduce_mean(tf.abs(cropped/255-globals()[f'resized_20_{method}']))
        MAE_20_matched = tf.reduce_mean(tf.abs(cropped/255-matched_20))

        # Resize images from 25cm back to 15cm and compute IQA metrics
        img=globals()[f'resized_25_{method}'].numpy()
        matched_25 = match_histograms(img, ref, channel_axis= -1)

        SSIM_25 = tf.image.ssim(cropped/255, globals()[f'resized_25_{method}'], max_val=1.0,filter_size=9,
                              filter_sigma=1, k1=0.01, k2=0.03)
        SSIM_25_matched = tf.image.ssim(cropped/255, matched_25, max_val=1.0,filter_size=9,
                              filter_sigma=1, k1=0.01, k2=0.03)
        PSNR_25 = tf.image.psnr(cropped/255, globals()[f'resized_25_{method}'], max_val = 1.0)
        PSNR_25_matched = tf.image.psnr(cropped/255, matched_25, max_val = 1.0)
        MSE_25 = skimage.metrics.mean_squared_error(ref,img)
        MSE_25_matched = skimage.metrics.mean_squared_error(ref,matched_25)
        MAE_25 = tf.reduce_mean(tf.abs(cropped/255-globals()[f'resized_25_{method}']))
        MAE_25_matched = tf.reduce_mean(tf.abs(cropped/255-matched_25))

        globals()[f'stat_{method}_20'] = [float(SSIM_20),float(PSNR_20),float(MSE_20),float(MAE_20)]
        globals()[f'stat_{method}_20_matched'] = [float(SSIM_20_matched),float(PSNR_20_matched),float(MSE_20_matched),float(MAE_20_matched)]
        globals()[f'stat_{method}_25'] = [float(SSIM_25),float(PSNR_25),float(MSE_25),float(MAE_25)]
        globals()[f'stat_{method}_25_matched'] = [float(SSIM_25_matched),float(PSNR_25_matched),float(MSE_25_matched),float(MAE_25_matched)]

        metrics.append(globals()[f'stat_{method}_20'])
        metrics.append(globals()[f'stat_{method}_20_matched'])
        metrics.append(globals()[f'stat_{method}_25'])
        metrics.append(globals()[f'stat_{method}_25_matched'])

        return metrics

all_metrics = []
for i in range(len(img_name)):
    metrics = IQA(img_name[i], img_x[i], img_y[i])
    all_metrics.extend(metrics)

methods = ['nearest', 'bicubic', 'lanczos3', 'lanczos5', 'gaussian']
matched = ['no', 'yes']

data = []

for i, m in enumerate(all_metrics):
    method_idx = (i // 4) % 5
    matched_flag = matched[i % 2]
    data.append({
        'method': methods[method_idx],
        'matched': matched_flag,
        'SSIM': m[0],
        'PSNR': m[1],
        'MSE': m[2],
        'MAE': m[3]
    })

df = pd.DataFrame(data)
df_long = df.melt(id_vars=['method', 'matched'],
                  var_name='metric', value_name='value')

summary = (
    df_long
    .groupby(['metric', 'method', 'matched'])['value']
    .agg(['mean', 'std'])
    .reset_index()
)

summary['mean'] = summary['mean'].round(4)
summary['std'] = summary['std'].round(4)

print(summary)

# **3. YOLO11 training**

**3.1. Importing prerequisites**

In [ ]:
from ultralytics import YOLO
import torch
import os

**3.2. Checks CUDA availability**

In [ ]:
print(f"Is CUDA supported? {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name()}")

**3.3. Training setup**

All combinations of model size, optimizer, initialization and batch size are trained for 100 epochs and compared on the validation subset. Pretrained models start from the COCO weights (`yolo11*.pt`); models trained from scratch are built from the architecture file (`yolo11*.yaml`) with randomly initialized weights. The other hyperparameters, including the online augmentation of Ultralytics (random rescaling, translation, horizontal flips, mosaic, and hue, saturation and brightness jitter), keep their default values.

In [ ]:
model_sizes = ["n", "s", "m", "l", "x"] #Model variants
optimizers = ["AdamW","SGD"] #Optimizers
pretrained_options = [True,False] #True: COCO-pretrained weights, False: training from scratch
batch_sizes = [8, 16, 32] #Batch sizes
data_path = "/content/GMAI/SAMPLE_DATASET/final_dataset/dataset.yaml" #Path to YAML file of dataset
project_name = "training"
epochs = 100 #Epoch number
img_size = 1280
if torch.cuda.is_available(): #If CUDA is available, chooses GPU for training
  device = [0]
else:
  device = 'cpu'

**3.4. Training loop**

In [ ]:
for size in model_sizes:
    for optimizer in optimizers:
        for pretrained in pretrained_options:
            for batch_size in batch_sizes:
                run_name = f"YOLO11_{size}_{optimizer}_bs{batch_size}_{'pretrained' if pretrained else 'scratch'}"
                print(f"\nTraining YOLO11{size} | {optimizer} | Pretrained: {pretrained} | Batch: {batch_size} ...")

                if pretrained:
                    model = YOLO(f"yolo11{size}.pt")    # COCO-pretrained weights
                else:
                    model = YOLO(f"yolo11{size}.yaml")  # same architecture, randomly initialized

                results = model.train(
                    data=data_path,
                    epochs=epochs,
                    imgsz=img_size,
                    batch=batch_size,
                    device=device,
                    project=project_name,
                    name=run_name,
                    optimizer=optimizer,
                    pretrained=pretrained,
                    seed=0,
                    deterministic=True,
                    val=True,
                    verbose=True
                )

                torch.cuda.empty_cache()

**3.5. Extended training of the selected configuration and evaluation on the test subset**

The configuration with the best validation metrics (in the study: YOLO11x, COCO-pretrained, SGD, batch size 16) is trained again with early stopping, which ends training after 30 epochs without improvement on the validation subset. The final model is then evaluated once on the test subset, which is used neither for training nor for model selection.

In [ ]:
best_size, best_optimizer, best_pretrained, best_batch = "x", "SGD", True, 16  # selected on the validation subset

model = YOLO(f"yolo11{best_size}.pt" if best_pretrained else f"yolo11{best_size}.yaml")
model.train(
    data=data_path,
    epochs=1000,
    patience=30,
    imgsz=img_size,
    batch=best_batch,
    device=device,
    project=project_name,
    name="final",
    optimizer=best_optimizer,
    pretrained=best_pretrained,
    seed=0,
    deterministic=True
)

final_model = YOLO(os.path.join(project_name, "final", "weights", "best.pt"))
test_metrics = final_model.val(data=data_path, split="test", imgsz=img_size, batch=best_batch, device=device)
print(f"Test subset: mAP50 {test_metrics.box.map50:.3f}, mAP50-95 {test_metrics.box.map:.3f}")

# **4. Inference with trained model**

**4.1. Importing prerequisites**

In [ ]:
from ultralytics import YOLO
import torch
import os
import shutil
from pathlib import Path
import rasterio
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import Polygon
import pandas as pd
import numpy as np
from rasterio.crs import CRS
import tensorflow as tf
from google.colab import files

**4.2. Checks CUDA availability**

In [ ]:
print(f"Is CUDA supported? {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name()}")

if torch.cuda.is_available(): #If CUDA is available, chooses GPU for predictions
  device = [0]
else:
  device = 'cpu'

**4.3. Find images for predictions**

In [ ]:
img_path = r"/content/GMAI/SAMPLE_ORTO"
out_path = r"/content/GMAI/predictions"

os.makedirs(out_path,exist_ok=True)

all_polygons, all_classes, all_confidences = [], [], []

tif_files = []

for dirpath, _, filenames in os.walk(img_path):
    for file in filenames:
        if file.lower().endswith(('.tif', '.tiff')):
            full_path = os.path.join(dirpath, file)
            tif_files.append(full_path)

print("Images found:",len(tif_files))

**4.3. Inference loop**

Each image is cut into tiles that cover the same ground area as the training tiles (192 × 192 m, i.e. 1280 px at 15 cm, 960 px at 20 cm and 768 px at 25 cm) with 10% overlap. Tiles of imagery coarser than 15 cm are resized to 1280 × 1280 px (15 cm pixel size) with Lanczos5 interpolation before prediction.

In [ ]:
TILE_GROUND_SIZE = 1280 * 0.15  # ground size of a training tile (m)
model = YOLO("/content/GMAI/YOLO11x_GMAI.pt")

for image in tif_files:

    image_folder = os.path.join(out_path, os.path.basename(os.path.splitext(image)[0]))
    os.makedirs(image_folder,exist_ok=True)

    tile_folder = os.path.join(image_folder, "tile")
    os.makedirs(tile_folder,exist_ok=True)

    #Tile
    with rasterio.open(image) as src:
        crs = src.crs
        image_width = src.width
        image_height = src.height
        tiling_size = int(round(TILE_GROUND_SIZE / abs(src.transform.a)))  # 1280 px at 15 cm, 960 at 20 cm, 768 at 25 cm
        overlap = 0.1
        stride_x, stride_y = int(tiling_size * (1 - overlap)), int(tiling_size * (1 - overlap))

        tile_id=0

        for top in range(0, image_height, stride_y):
            for left in range(0, image_width, stride_x):
                if left + tiling_size > image_width:
                    left = image_width - tiling_size
                if top + tiling_size > image_height:
                    top = image_height - tiling_size
                window = Window(left, top, tiling_size, tiling_size)
                tile_path = os.path.join(tile_folder, f"{os.path.basename(os.path.splitext(image)[0])}_{tile_id}.tif")
                tile_id+=1
                tile_data = src.read(indexes=[1, 2, 3],window=window)
                with rasterio.open(
                                tile_path,
                                "w",
                                driver="GTiff",
                                height=tile_data.shape[1],
                                width=tile_data.shape[2],
                                count=3,
                                dtype=tile_data.dtype,
                                crs=crs,
                                transform=src.window_transform(window),
                            ) as dst:
                                dst.write(tile_data)

    #Resize tiles of coarser imagery to the 15 cm pixel size of the training data (Lanczos5)
    if tiling_size != 1280:
        for tile_name in os.listdir(tile_folder):
            tile_path = os.path.join(tile_folder, tile_name)
            with rasterio.open(tile_path) as t:
                data = np.moveaxis(t.read(), 0, -1)
                profile = t.profile
                transform = t.transform
            resized = tf.image.resize(data, [1280, 1280], method="lanczos5").numpy()
            resized = np.clip(resized, 0, 255).astype(np.uint8)
            scale = data.shape[1] / 1280
            profile.update(width=1280, height=1280, transform=transform * transform.scale(scale, scale))
            with rasterio.open(tile_path, "w", **profile) as dst:
                dst.write(np.moveaxis(resized, -1, 0))

    results = model.predict(
        source=tile_folder,
        conf=0.3,
        imgsz=1280,
        save_txt=True,
        save_conf=True,
        project = image_folder,
        name='predict')

    for result in results:
            image_path = Path(result.path)
            image_name = image_path.stem
            boxes = result.boxes

            if boxes is not None and len(boxes) > 0:
                result_path = os.path.join(image_folder, f"{image_name}.tif")
                result.save(filename=result_path)

                txt_filename = f"{image_name}.txt"
                txt_source_path = os.path.join(image_folder,"predict/labels", txt_filename)  # Default YOLO output folder
                txt_dest_path = os.path.join(image_folder,'labels', txt_filename)
                os.makedirs(os.path.dirname(txt_dest_path), exist_ok=True)
                if os.path.exists(txt_source_path):
                    shutil.move(txt_source_path, txt_dest_path)

    shutil.rmtree(os.path.join(image_folder,"predict"))

**4.5. Shapefile creation**

In [ ]:
for image in tif_files:
  image_folder = os.path.join(out_path, os.path.basename(os.path.splitext(image)[0]))
  tile_folder = os.path.join(image_folder, "tile")
  if not os.path.isdir(os.path.join(image_folder,'labels')):
    continue
  for file in os.listdir(os.path.join(image_folder,'labels')):
    df = pd.read_csv(os.path.join(image_folder,'labels',file), delimiter=' ', header=None, names=["class_id", "cx", "cy", "w", "h", "conf"])
    tile_path = os.path.join(tile_folder,f"{os.path.basename(os.path.splitext(file)[0])}.tif")

    with rasterio.open(tile_path) as src:
      transform = src.transform
      crs = src.crs
      img_w, img_h = src.width, src.height
      df[["cx", "w"]] *= img_w
      df[["cy", "h"]] *= img_h
      x_min, x_max = df["cx"] - df["w"] / 2, df["cx"] + df["w"] / 2
      y_min, y_max = df["cy"] - df["h"] / 2, df["cy"] + df["h"] / 2
      bbox_coords = np.column_stack([x_min, y_min, x_min, y_max, x_max, y_max, x_max, y_min, x_min, y_min])
      bbox_coords = bbox_coords.reshape(-1, 5, 2)
      transformed_coords = np.apply_along_axis(lambda pt: transform * tuple(pt), 2, bbox_coords)

      polygons = [Polygon(coords) for coords in transformed_coords]

      all_polygons.extend(polygons)
      all_classes.extend(df["class_id"].astype(int))
      all_confidences.extend(df["conf"])

gdf = gpd.GeoDataFrame(
  {"geometry": all_polygons, "class_id": all_classes, "confidence": all_confidences},
   crs=crs
    )

gdf.to_file(os.path.join(out_path, "predictions.shp"), driver="ESRI Shapefile")

**4.6. Download results**

In [ ]:
zip_filename = "predictions.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', out_path)
files.download(zip_filename)

print(f"Folder '{out_path}' compressed to '{zip_filename}' and downloaded.")